First step: Align/Rotate the image so it is 180° for all cells.
Next: Isolate cells of interest by cropping and blacking out other cells.
Finally: Run this Function

In [ ]:
def Threshold_Test(folder_path):
    import os
    from skimage import io
    from skimage.filters import try_all_threshold
    import matplotlib.pyplot as plt
    """Test threshold options for images in a folder

    Parameters
    ----------
    folder_path : str
        Path to the folder containing the images

    Returns
    -------
    None
    """
    # Get a list of all image files in the folder
    image_files = [file for file in os.listdir(folder_path) if file.endswith('.tiff') or file.endswith('.tif')]

    # Iterate over each image file
    for image_file in image_files:
        # Construct the full path to the image file
        image_path = os.path.join(folder_path, image_file)

        # Read the image
        img = io.imread(image_path)
        fig, ax = try_all_threshold(img, figsize=(20, 18), verbose=False)
        plt.title(image_file)  # Add the image file name as the title
        plt.show()

Threshold_Test('/Users/allisonpickle/Desktop/Test')

In [ ]:
from skimage.filters import threshold_mean
from skimage.io import imread, imsave
from skimage.morphology import binary_closing, remove_small_objects, disk
import os
import matplotlib.pyplot as plt

def apply_mean_threshold(folder_path, display_results=False, save_results=True, min_size_remove=64, min_size_fill=64):
    """
    Apply mean thresholding to all TIFF images in the specified folder and process small objects.
    
    Parameters
    ----------
    folder_path : str
        The path to the folder containing the TIFF images.
    display_results : bool, optional
        Whether to display the thresholded images. Default is False.
    save_results : bool, optional
        Whether to save the thresholded images. Default is True.
    min_size_remove : int, optional
        The minimum size of small objects to remove. Default is 64.
    min_size_fill : int, optional
        The minimum size of small holes to fill. Default is 64.
    """
    # Create the output folder if it doesn't exist
    output_folder = os.path.join(folder_path, "0_Mean_Threshold")
    os.makedirs(output_folder, exist_ok=True)

    # List all TIFF files in the folder
    tiff_files = [f for f in os.listdir(folder_path) if f.endswith('.tif') or f.endswith('.tiff')]

    for tiff_file in tiff_files:
        # Read the image
        image_path = os.path.join(folder_path, tiff_file)
        image = imread(image_path, as_gray=True)

        # Apply mean thresholding
        thresh = threshold_mean(image)
        binary_image = image > thresh

        # Remove small objects
        binary_image = remove_small_objects(binary_image, min_size=min_size_remove)

        # Fill small holes
        binary_image = binary_closing(binary_image, disk(1))
        binary_image = remove_small_objects(~binary_image, min_size=min_size_fill)
        binary_image = ~binary_image

        if display_results:
            # Display the original and thresholded images
            fig, ax = plt.subplots(1, 2, figsize=(12, 6))
            ax[0].imshow(image, cmap='gray')
            ax[0].set_title('Original Image')
            ax[0].axis('off')
            ax[1].imshow(binary_image, cmap='gray')
            ax[1].set_title('Mean Thresholded Image')
            ax[1].axis('off')
            plt.show()

        if save_results:
            # Save the thresholded image
            save_path = os.path.join(output_folder, tiff_file)
            imsave(save_path, binary_image.astype('uint8') * 255)
            print(f"Saved {save_path}")

# Example usage
folder_path = '/Users/allisonpickle/Desktop/True Neuros'
apply_mean_threshold(folder_path, display_results=False, save_results=True, min_size_remove=25, min_size_fill=100)

In [27]:
#Otsu for Astrocytes and probs neurons


from skimage.filters import threshold_otsu
from skimage.io import imread, imsave
from skimage.morphology import binary_closing, remove_small_objects, disk
import os
import matplotlib.pyplot as plt

def apply_otsu_threshold(folder_path, display_results=False, save_results=True, min_size_remove=64, min_size_fill=64):
    """
    Apply Otsu's thresholding to all TIFF images in the specified folder and process small objects.
    
    Parameters
    ----------
    folder_path : str
        The path to the folder containing the TIFF images.
    display_results : bool, optional
        Whether to display the thresholded images. Default is False.
    save_results : bool, optional
        Whether to save the thresholded images. Default is True.
    min_size_remove : int, optional
        The minimum size of small objects to remove. Default is 64.
    min_size_fill : int, optional
        The minimum size of small holes to fill. Default is 64.
    """
    # Create the output folder if it doesn't exist
    output_folder = os.path.join(folder_path, "0_Threshold")
    os.makedirs(output_folder, exist_ok=True)

    # List all TIFF files in the folder
    tiff_files = [f for f in os.listdir(folder_path) if f.endswith('.tif')]

    for tiff_file in tiff_files:
        # Read the image
        image_path = os.path.join(folder_path, tiff_file)
        image = imread(image_path, as_gray=True)

        # Apply Otsu's thresholding
        thresh = threshold_otsu(image)
        binary_image = image > thresh

        # Remove small objects
        binary_image = remove_small_objects(binary_image, min_size=min_size_remove)

        # Fill small holes
        binary_image = binary_closing(binary_image, disk(1))
        binary_image = remove_small_objects(~binary_image, min_size=min_size_fill)
        binary_image = ~binary_image

        if display_results:
            # Display the original and thresholded images
            fig, ax = plt.subplots(1, 2, figsize=(12, 6))
            ax[0].imshow(image, cmap='gray')
            ax[0].set_title('Original Image')
            ax[0].axis('off')
            ax[1].imshow(binary_image, cmap='gray')
            ax[1].set_title('Otsu Thresholded Image')
            ax[1].axis('off')
            plt.show()

        if save_results:
            # Save the thresholded image
            save_path = os.path.join(output_folder, tiff_file)
            imsave(save_path, binary_image.astype('uint8') * 255)
            print(f"Saved {save_path}")

# Example usage
folder_path = '/Users/allisonpickle/Desktop/Test'
apply_otsu_threshold(folder_path, display_results=False, save_results=True, min_size_remove=10, min_size_fill=10)

Saved /Users/allisonpickle/Desktop/Test/0_Threshold/unnamed-1.tif


Live Imaging Processing:

In [ ]:
import numpy as np
from scipy.ndimage import center_of_mass
from skimage import io
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import os

def calculate_inertia_tensor(binary_image):
    """
    Calculate the inertia tensor of a thresholded (binary) image based on its center of mass.

    Parameters
    ----------
    binary_image : np.ndarray
        The input binary image as a 2D numpy array.

    Returns
    -------
    np.ndarray
        The 2x2 inertia tensor of the thresholded image.
    """
    # Calculate the center of mass of the binary image
    com = center_of_mass(binary_image)

    # Get the coordinates of all pixels in the image
    y_indices, x_indices = np.indices(binary_image.shape)

    # Calculate the distances of all pixels from the center of mass
    x_distances = x_indices - com[1]
    y_distances = y_indices - com[0]

    # Calculate the elements of the inertia tensor
    Ixx = np.sum(binary_image * y_distances**2)
    Iyy = np.sum(binary_image * x_distances**2)
    Ixy = -np.sum(binary_image * x_distances * y_distances)

    # Construct the inertia tensor
    inertia_tensor = np.array([[Ixx, Ixy], [Ixy, Iyy]])

    return inertia_tensor

def calculate_principal_moments_and_axes(inertia_tensor):
    """
    Calculate the principal moments of inertia and their corresponding eigenvectors.

    Parameters
    ----------
    inertia_tensor : np.ndarray
        The 2x2 inertia tensor.

    Returns
    -------
    np.ndarray
        The eigenvalues of the inertia tensor, representing the principal moments of inertia.
    np.ndarray
        The eigenvectors of the inertia tensor, representing the principal axes.
    """
    # Calculate the eigenvalues and eigenvectors of the inertia tensor
    eigenvalues, eigenvectors = np.linalg.eigh(inertia_tensor)

    return eigenvalues, eigenvectors

def calculate_total_moment_of_inertia(binary_image):
    """
    Calculate the total moment of inertia of a thresholded (binary) image based on its center of mass.

    Parameters
    ----------
    binary_image : np.ndarray
        The input binary image as a 2D numpy array.

    Returns
    -------
    float
        The total moment of inertia of the thresholded image.
    """
    # Calculate the center of mass of the binary image
    com = center_of_mass(binary_image)

    # Get the coordinates of all pixels in the image
    y_indices, x_indices = np.indices(binary_image.shape)

    # Calculate the distances of all pixels from the center of mass
    distances_squared = (x_indices - com[1])**2 + (y_indices - com[0])**2

    # Calculate the total moment of inertia
    total_moment_of_inertia = np.sum(binary_image * distances_squared)

    return total_moment_of_inertia

def calculate_orientation_angle(eigenvector):
    """
    Calculate the orientation angle theta as the angle formed between the long axis of the cell
    and a straight 0° line, mirroring angles between 90° and 180° to be between 0° and 90°.

    Parameters
    ----------
    eigenvector : np.ndarray
        The eigenvector corresponding to the smaller eigenvalue.

    Returns
    -------
    float
        The orientation angle theta in degrees.
    """
    # Calculate the angle in radians
    theta_radians = np.arctan2(eigenvector[1], eigenvector[0])

    # Convert the angle to degrees
    theta_degrees = np.degrees(theta_radians)

    # Ensure the angle is positive
    if theta_degrees < 0:
        theta_degrees += 360

    # Mirror the angle if it is between 90° and 180°
    if 90 < theta_degrees <= 180:
        theta_degrees = 180 - theta_degrees
    elif 180 < theta_degrees <= 270:
        theta_degrees = theta_degrees - 180
    elif 270 < theta_degrees < 360:
        theta_degrees = 360 - theta_degrees

    return theta_degrees

def calculate_aspect_ratio(eigenvalues):
    """
    Calculate the aspect ratio of the cell as the square root of the ratio of the large to the small eigenvalue.

    Parameters
    ----------
    eigenvalues : np.ndarray
        The eigenvalues of the inertia tensor.

    Returns
    -------
    float
        The aspect ratio of the cell.
    """
    # Calculate the aspect ratio
    aspect_ratio = np.sqrt(np.max(eigenvalues) / np.min(eigenvalues))

    return aspect_ratio

def calculate_cell_size(binary_image, pixel_area=0.1123):
    """
    Calculate the size of a cell/cell cluster as the number of pixels times the area of a pixel.

    Parameters
    ----------
    binary_image : np.ndarray
        The input binary image as a 2D numpy array.
    pixel_area : float, optional
        The area of a single pixel in µm². Default is 0.1123 µm².

    Returns
    -------
    float
        The size of the cell/cell cluster in µm².
    """
    # Count the number of pixels in the binary image
    num_pixels = np.sum(binary_image)

    # Calculate the size of the cell/cell cluster
    cell_size = num_pixels * pixel_area

    return cell_size

def display_contour(binary_image, image_name, folder_path, show_contour=True):
    """
    Display a contour around the measured area for cell size and save the contour image.

    Parameters
    ----------
    binary_image : np.ndarray
        The input binary image as a 2D numpy array.
    image_name : str
        The name of the image file.
    folder_path : str
        The path to the main folder where the results will be saved.
    show_contour : bool, optional
        Whether to display the contour image. Default is True.
    """
    # Find contours in the binary image
    contours, _ = cv2.findContours(binary_image.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Check if any contours were found
    if len(contours) == 0:
        print(f"No contours found for {image_name}.")
        return

    # Create an RGB version of the binary image to draw colored contours
    contour_image = cv2.cvtColor(binary_image.astype(np.uint8) * 255, cv2.COLOR_GRAY2RGB)

    # Draw contours on the binary image
    cv2.drawContours(contour_image, contours, -1, (255, 105, 180), 2)  # Pink color for contours

    # Save the contour image to the "Contour Regions" folder within the main folder
    contour_folder = os.path.join(folder_path, "Contour Regions")
    os.makedirs(contour_folder, exist_ok=True)
    contour_image_path = os.path.join(contour_folder, f"{image_name}.png")
    cv2.imwrite(contour_image_path, contour_image)

    # Display the image with contours if show_contour is True
    if show_contour:
        plt.figure(figsize=(10, 10))
        plt.imshow(contour_image)
        plt.title('Contour around the measured area')
        plt.axis('off')
        plt.show()

def plot_polar(all_aspect_ratios, all_orientation_angles, folder_names, save_path):
    """
    Plot the data on a polar plot with the distance to the origin (R) set to the aspect ratio - 1
    and the angle to the horizontal (theta) set to the orientation angle.

    Parameters
    ----------
    all_aspect_ratios : list of list of float
        The aspect ratios of the cells for each folder.
    all_orientation_angles : list of list of float
        The orientation angles of the cells in degrees for each folder.
    folder_names : list of str
        The names of the folders.
    save_path : str
        The path where the plot will be saved.
    """
    colors = plt.cm.rainbow(np.linspace(0, 1, len(folder_names)))

    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)

    for i, (aspect_ratios, orientation_angles) in enumerate(zip(all_aspect_ratios, all_orientation_angles)):
        R = [aspect_ratio - 1 for aspect_ratio in aspect_ratios]
        theta = [np.radians(angle) for angle in orientation_angles]
        ax.scatter(theta, R, label=folder_names[i], color=colors[i], s=5)  # Adjust marker size as needed

    ax.set_title('Polar Plot of Cell Orientation and Aspect Ratio')
    ax.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1))
    plt.savefig(save_path)
    plt.show()

def plot_polar_zoomed(all_aspect_ratios, all_orientation_angles, folder_names, save_path):
    """
    Plot the zoomed-in data on a polar plot for the 0 to 90° region with the distance to the origin (R) set to the aspect ratio - 1
    and the angle to the horizontal (theta) set to the orientation angle.

    Parameters
    ----------
    all_aspect_ratios : list of list of float
        The aspect ratios of the cells for each folder.
    all_orientation_angles : list of list of float
        The orientation angles of the cells in degrees for each folder.
    folder_names : list of str
        The names of the folders.
    save_path : str
        The path where the plot will be saved.
    """
    colors = plt.cm.rainbow(np.linspace(0, 1, len(folder_names)))

    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)

    for i, (aspect_ratios, orientation_angles) in enumerate(zip(all_aspect_ratios, all_orientation_angles)):
        R = [aspect_ratio - 1 for aspect_ratio in aspect_ratios]
        theta = [np.radians(angle) for angle in orientation_angles]
        ax.scatter(theta, R, label=folder_names[i], color=colors[i], s=15)  # Adjust marker size as needed

    ax.set_ylim(0, max(max(R) for R in all_aspect_ratios))
    ax.set_xlim(0, np.radians(90))
    ax.set_title('Polar Plot of Cell Orientation and Aspect Ratio (0 to 90°)')
    ax.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1))
    plt.savefig(save_path)
    plt.show()

def process_images_in_folder(folder_path):
    """
    Process all images in the specified folder and save the results to the main folder path.

    Parameters
    ----------
    folder_path : str
        The path to the folder containing the images to be processed and where the results will be saved.
    """
    all_aspect_ratios = []
    all_orientation_angles = []
    folder_names = []
    calculations = []

    for subfolder in os.listdir(folder_path):
        subfolder_path = os.path.join(folder_path, subfolder)
        if os.path.isdir(subfolder_path):
            aspect_ratios = []
            orientation_angles = []

            for image_file in os.listdir(subfolder_path):
                if image_file.endswith(('.tif', '.png', '.jpg', '.jpeg', '.tiff')):
                    image_path = os.path.join(subfolder_path, image_file)
                    image_name = os.path.splitext(image_file)[0]

                    # Load the thresholded image
                    binary_image = io.imread(image_path, as_gray=True)

                    # Ensure the image is binary (0 and 1)
                    binary_image = binary_image > 0.5

                    # Calculate the inertia tensor
                    inertia_tensor = calculate_inertia_tensor(binary_image)

                    # Calculate the principal moments of inertia and their corresponding eigenvectors
                    principal_moments, eigenvectors = calculate_principal_moments_and_axes(inertia_tensor)

                    # Calculate the total moment of inertia
                    total_moment_of_inertia = calculate_total_moment_of_inertia(binary_image)

                    # Assign the eigenvector corresponding to the smaller eigenvalue as the long axis
                    long_axis = eigenvectors[:, np.argmin(principal_moments)]

                    # Calculate the orientation angle theta
                    orientation_angle = calculate_orientation_angle(long_axis)

                    # Calculate the aspect ratio
                    aspect_ratio = calculate_aspect_ratio(principal_moments)

                    # Calculate the cell size
                    cell_size = calculate_cell_size(binary_image)

                    # Display the contour around the measured area and save the contour image
                    display_contour(binary_image, image_name, folder_path, show_contour=False)

                    # Append the aspect ratio and orientation angle to the lists
                    aspect_ratios.append(aspect_ratio)
                    orientation_angles.append(orientation_angle)

                    # Append the calculations to the list
                    calculations.append({
                        "Subfolder": subfolder,
                        "Image Name": image_name,
                        "Aspect Ratio": aspect_ratio,
                        "Orientation Angle": orientation_angle,
                        "Total Moment of Inertia": total_moment_of_inertia,
                        "Principal Moments": principal_moments.tolist(),
                        "Eigenvectors": eigenvectors.tolist(),
                        "Cell Size (µm²)": cell_size
                    })

            # Append the results for the current folder to the main lists
            all_aspect_ratios.append(aspect_ratios)
            all_orientation_angles.append(orientation_angles)
            folder_names.append(subfolder)

    # Save all results to a single Excel file in the main folder
    all_results = []
    for subfolder, aspect_ratios, orientation_angles in zip(folder_names, all_aspect_ratios, all_orientation_angles):
        for aspect_ratio, orientation_angle in zip(aspect_ratios, orientation_angles):
            all_results.append({
                "Subfolder": subfolder,
                "Aspect Ratio": aspect_ratio,
                "Orientation Angle": orientation_angle
            })
    df = pd.DataFrame(all_results)
    excel_file = os.path.join(folder_path, "Contour_regions.xlsx")
    df.to_excel(excel_file, index=False)

    # Save the calculations to a single Excel sheet
    calculations_df = pd.DataFrame(calculations)
    calculations_file = os.path.join(folder_path, "Calculations.xlsx")
    calculations_df.to_excel(calculations_file, index=False)

    # Plot all points on a polar plot and save it
    plot_polar(all_aspect_ratios, all_orientation_angles, folder_names, os.path.join(folder_path, "0_polar_plot.png"))

    # Plot the zoomed-in region (0 to 90°) on a polar plot and save it
    plot_polar_zoomed(all_aspect_ratios, all_orientation_angles, folder_names, os.path.join(folder_path, "0_polar_plot_zoomed.png"))

# Example usage
if __name__ == "__main__":
    # Define the folder path where images are located and results will be saved
    folder_path = '/Users/allisonpickle/Desktop/Test/0_Threshold'
    # Process all images in the folder and save the results
    process_images_in_folder(folder_path)

In [ ]:
import numpy as np
from scipy.ndimage import center_of_mass
from skimage import io
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import os

def calculate_inertia_tensor(binary_image):
    """
    Calculate the inertia tensor of a thresholded (binary) image based on its center of mass.

    Parameters
    ----------
    binary_image : np.ndarray
        The input binary image as a 2D numpy array.

    Returns
    -------
    np.ndarray
        The 2x2 inertia tensor of the thresholded image.
    """
    # Calculate the center of mass of the binary image
    com = center_of_mass(binary_image)

    # Get the coordinates of all pixels in the image
    y_indices, x_indices = np.indices(binary_image.shape)

    # Calculate the distances of all pixels from the center of mass
    x_distances = x_indices - com[1]
    y_distances = y_indices - com[0]

    # Calculate the elements of the inertia tensor
    Ixx = np.sum(binary_image * y_distances**2)
    Iyy = np.sum(binary_image * x_distances**2)
    Ixy = -np.sum(binary_image * x_distances * y_distances)

    # Construct the inertia tensor
    inertia_tensor = np.array([[Ixx, Ixy], [Ixy, Iyy]])

    return inertia_tensor

def calculate_principal_moments_and_axes(inertia_tensor):
    """
    Calculate the principal moments of inertia and their corresponding eigenvectors.

    Parameters
    ----------
    inertia_tensor : np.ndarray
        The 2x2 inertia tensor.

    Returns
    -------
    np.ndarray
        The eigenvalues of the inertia tensor, representing the principal moments of inertia.
    np.ndarray
        The eigenvectors of the inertia tensor, representing the principal axes.
    """
    # Calculate the eigenvalues and eigenvectors of the inertia tensor
    eigenvalues, eigenvectors = np.linalg.eigh(inertia_tensor)

    return eigenvalues, eigenvectors

def calculate_total_moment_of_inertia(binary_image):
    """
    Calculate the total moment of inertia of a thresholded (binary) image based on its center of mass.

    Parameters
    ----------
    binary_image : np.ndarray
        The input binary image as a 2D numpy array.

    Returns
    -------
    float
        The total moment of inertia of the thresholded image.
    """
    # Calculate the center of mass of the binary image
    com = center_of_mass(binary_image)

    # Get the coordinates of all pixels in the image
    y_indices, x_indices = np.indices(binary_image.shape)

    # Calculate the distances of all pixels from the center of mass
    distances_squared = (x_indices - com[1])**2 + (y_indices - com[0])**2

    # Calculate the total moment of inertia
    total_moment_of_inertia = np.sum(binary_image * distances_squared)

    return total_moment_of_inertia

def calculate_orientation_angle(eigenvector):
    """
    Calculate the orientation angle theta as the angle formed between the long axis of the cell
    and a straight 0° line, mirroring angles between 90° and 180° to be between 0° and 90°.

    Parameters
    ----------
    eigenvector : np.ndarray
        The eigenvector corresponding to the smaller eigenvalue.

    Returns
    -------
    float
        The orientation angle theta in degrees.
    """
    # Calculate the angle in radians
    theta_radians = np.arctan2(eigenvector[1], eigenvector[0])

    # Convert the angle to degrees
    theta_degrees = np.degrees(theta_radians)

    # Ensure the angle is positive
    if theta_degrees < 0:
        theta_degrees += 360

    # Mirror the angle if it is between 90° and 180°
    if 90 < theta_degrees <= 180:
        theta_degrees = 180 - theta_degrees
    elif 180 < theta_degrees <= 270:
        theta_degrees = theta_degrees - 180
    elif 270 < theta_degrees < 360:
        theta_degrees = 360 - theta_degrees

    return theta_degrees

def calculate_aspect_ratio(eigenvalues):
    """
    Calculate the aspect ratio of the cell as the square root of the ratio of the large to the small eigenvalue.

    Parameters
    ----------
    eigenvalues : np.ndarray
        The eigenvalues of the inertia tensor.

    Returns
    -------
    float
        The aspect ratio of the cell.
    """
    # Calculate the aspect ratio
    aspect_ratio = np.sqrt(np.max(eigenvalues) / np.min(eigenvalues))

    return aspect_ratio

def calculate_cell_size(binary_image, pixel_area=0.1123):
    """
    Calculate the size of a cell/cell cluster as the number of pixels times the area of a pixel.

    Parameters
    ----------
    binary_image : np.ndarray
        The input binary image as a 2D numpy array.
    pixel_area : float, optional
        The area of a single pixel in µm². Default is 0.1123 µm².

    Returns
    -------
    float
        The size of the cell/cell cluster in µm².
    """
    # Count the number of pixels in the binary image
    num_pixels = np.sum(binary_image)

    # Calculate the size of the cell/cell cluster
    cell_size = num_pixels * pixel_area

    return cell_size

def display_contour(binary_image, image_name, folder_path, show_contour=True):
    """
    Display a contour around the measured area for cell size and save the contour image.

    Parameters
    ----------
    binary_image : np.ndarray
        The input binary image as a 2D numpy array.
    image_name : str
        The name of the image file.
    folder_path : str
        The path to the main folder where the results will be saved.
    show_contour : bool, optional
        Whether to display the contour image. Default is True.
    """
    # Find contours in the binary image
    contours, _ = cv2.findContours(binary_image.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Check if any contours were found
    if len(contours) == 0:
        print(f"No contours found for {image_name}.")
        return

    # Create an RGB version of the binary image to draw colored contours
    contour_image = cv2.cvtColor(binary_image.astype(np.uint8) * 255, cv2.COLOR_GRAY2RGB)

    # Draw contours on the binary image
    cv2.drawContours(contour_image, contours, -1, (255, 105, 180), 2)  # Pink color for contours

    # Save the contour image to the "Contour Regions" folder within the main folder
    contour_folder = os.path.join(folder_path, "Contour Regions")
    os.makedirs(contour_folder, exist_ok=True)
    contour_image_path = os.path.join(contour_folder, f"{image_name}.png")
    cv2.imwrite(contour_image_path, contour_image)

    # Display the image with contours if show_contour is True
    if show_contour:
        plt.figure(figsize=(10, 10))
        plt.imshow(contour_image)
        plt.title('Contour around the measured area')
        plt.axis('off')
        plt.show()

def plot_polar(all_aspect_ratios, all_orientation_angles, folder_names, save_path):
    """
    Plot the data on a polar plot with the distance to the origin (R) set to the aspect ratio - 1
    and the angle to the horizontal (theta) set to the orientation angle.

    Parameters
    ----------
    all_aspect_ratios : list of list of float
        The aspect ratios of the cells for each folder.
    all_orientation_angles : list of list of float
        The orientation angles of the cells in degrees for each folder.
    folder_names : list of str
        The names of the folders.
    save_path : str
        The path where the plot will be saved.
    """
    colors = plt.cm.rainbow(np.linspace(0, 1, len(folder_names)))

    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)

    for i, (aspect_ratios, orientation_angles) in enumerate(zip(all_aspect_ratios, all_orientation_angles)):
        R = [aspect_ratio - 1 for aspect_ratio in aspect_ratios]
        theta = [np.radians(angle) for angle in orientation_angles]
        ax.scatter(theta, R, label=folder_names[i], color=colors[i], s=5)  # Adjust marker size as needed

    ax.set_title('Polar Plot of Cell Orientation and Aspect Ratio')
    ax.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1))
    plt.savefig(save_path)
    plt.show()

def plot_polar_zoomed(all_aspect_ratios, all_orientation_angles, folder_names, save_path):
    """
    Plot the zoomed-in data on a polar plot for the 0 to 90° region with the distance to the origin (R) set to the aspect ratio - 1
    and the angle to the horizontal (theta) set to the orientation angle.

    Parameters
    ----------
    all_aspect_ratios : list of list of float
        The aspect ratios of the cells for each folder.
    all_orientation_angles : list of list of float
        The orientation angles of the cells in degrees for each folder.
    folder_names : list of str
        The names of the folders.
    save_path : str
        The path where the plot will be saved.
    """
    colors = plt.cm.rainbow(np.linspace(0, 1, len(folder_names)))

    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)

    for i, (aspect_ratios, orientation_angles) in enumerate(zip(all_aspect_ratios, all_orientation_angles)):
        R = [aspect_ratio - 1 for aspect_ratio in aspect_ratios]
        theta = [np.radians(angle) for angle in orientation_angles]
        ax.scatter(theta, R, label=folder_names[i], color=colors[i], s=15)  # Adjust marker size as needed

    ax.set_ylim(0, max(max(R) for R in all_aspect_ratios))
    ax.set_xlim(0, np.radians(90))
    ax.set_title('Polar Plot of Cell Orientation and Aspect Ratio (0 to 90°)')
    ax.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1))
    plt.savefig(save_path)
    plt.show()

def process_images_in_folder(folder_path):
    """
    Process all images in the specified folder and save the results to the main folder path.

    Parameters
    ----------
    folder_path : str
        The path to the folder containing the images to be processed and where the results will be saved.
    """
    all_aspect_ratios = []
    all_orientation_angles = []
    folder_names = []
    calculations = []

    for subfolder in os.listdir(folder_path):
        subfolder_path = os.path.join(folder_path, subfolder)
        if os.path.isdir(subfolder_path):
            aspect_ratios = []
            orientation_angles = []

            for image_file in os.listdir(subfolder_path):
                if image_file.endswith(('.tif', '.png', '.jpg', '.jpeg', '.tiff')):
                    image_path = os.path.join(subfolder_path, image_file)
                    image_name = os.path.splitext(image_file)[0]

                    # Load the thresholded image
                    binary_image = io.imread(image_path, as_gray=True)

                    # Ensure the image is binary (0 and 1)
                    binary_image = binary_image > 0.5

                    # Calculate the inertia tensor
                    inertia_tensor = calculate_inertia_tensor(binary_image)

                    # Calculate the principal moments of inertia and their corresponding eigenvectors
                    principal_moments, eigenvectors = calculate_principal_moments_and_axes(inertia_tensor)

                    # Calculate the total moment of inertia
                    total_moment_of_inertia = calculate_total_moment_of_inertia(binary_image)

                    # Assign the eigenvector corresponding to the smaller eigenvalue as the long axis
                    long_axis = eigenvectors[:, np.argmin(principal_moments)]

                    # Calculate the orientation angle theta
                    orientation_angle = calculate_orientation_angle(long_axis)

                    # Calculate the aspect ratio
                    aspect_ratio = calculate_aspect_ratio(principal_moments)

                    # Calculate the cell size
                    cell_size = calculate_cell_size(binary_image)

                    # Display the contour around the measured area and save the contour image
                    display_contour(binary_image, image_name, folder_path, show_contour=False)

                    # Append the aspect ratio and orientation angle to the lists
                    aspect_ratios.append(aspect_ratio)
                    orientation_angles.append(orientation_angle)

                    # Append the calculations to the list
                    calculations.append({
                        "Image Name": image_name,
                        "Aspect Ratio": aspect_ratio,
                        "Orientation Angle": orientation_angle,
                        "Total Moment of Inertia": total_moment_of_inertia,
                        "Principal Moments": principal_moments.tolist(),
                        "Eigenvectors": eigenvectors.tolist(),
                        "Cell Size (µm²)": cell_size
                    })

            # Append the results for the current folder to the main lists
            all_aspect_ratios.append(aspect_ratios)
            all_orientation_angles.append(orientation_angles)
            folder_names.append(subfolder)

    # Save all results to a single Excel file in the main folder
    all_results = []
    for subfolder, aspect_ratios, orientation_angles in zip(folder_names, all_aspect_ratios, all_orientation_angles):
        for aspect_ratio, orientation_angle in zip(aspect_ratios, orientation_angles):
            all_results.append({
                "Subfolder": subfolder,
                "Aspect Ratio": aspect_ratio,
                "Orientation Angle": orientation_angle
            })
    df = pd.DataFrame(all_results)
    excel_file = os.path.join(folder_path, "Contour_regions.xlsx")
    df.to_excel(excel_file, index=False)

    # Save the calculations to a single Excel sheet
    calculations_df = pd.DataFrame(calculations)
    calculations_file = os.path.join(folder_path, "Calculations.xlsx")
    calculations_df.to_excel(calculations_file, index=False)

    # Plot all points on a polar plot and save it
    plot_polar(all_aspect_ratios, all_orientation_angles, folder_names, os.path.join(folder_path, "0_polar_plot.png"))

    # Plot the zoomed-in region (0 to 90°) on a polar plot and save it
    plot_polar_zoomed(all_aspect_ratios, all_orientation_angles, folder_names, os.path.join(folder_path, "0_polar_plot_zoomed.png"))

# Example usage
if __name__ == "__main__":
    # Define the folder path where images are located and results will be saved
    folder_path = '/Users/allisonpickle/Desktop/Test/0_Threshold'
    # Process all images in the folder and save the results
    process_images_in_folder(folder_path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.drawing.image import Image
from openpyxl.utils.dataframe import dataframe_to_rows
import os

def plot_histograms(main_folder_path):
    """
    Plot histograms of the second and third columns as two graphs in an Excel sheet.
    Plot per same subfolder number in the first column.

    Parameters
    ----------
    main_folder_path : str
        The path to the main folder containing the input Excel file and where the output Excel file will be saved.
    """
    # Define the path to the input Excel file
    excel_file = os.path.join(main_folder_path, "Contour_regions.xlsx")

    # Define the path to the output Excel file
    output_file = os.path.join(main_folder_path, "output_with_histograms.xlsx")

    # Load the Excel file into a DataFrame
    df = pd.read_excel(excel_file)

    # Create a new Excel workbook
    wb = Workbook()
    wb.remove(wb.active)  # Remove the default sheet

    # Group the DataFrame by the first column (subfolder number)
    grouped = df.groupby(df.columns[0])

    # Create a directory for temporary images
    temp_image_dir = os.path.join(main_folder_path, "temp_images")
    os.makedirs(temp_image_dir, exist_ok=True)

    # Iterate over each group
    for subfolder, group in grouped:
        # Create a new figure for the histograms
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))

        # Plot the histogram for the second column
        group.iloc[:, 1].plot(kind='hist', bins=30, ax=axes[0], alpha=0.6, color='blue')
        axes[0].set_title(f'Subfolder {subfolder} - Column 2')
        axes[0].set_xlabel('Value')
        axes[0].set_ylabel('Frequency')

        # Plot the histogram for the third column
        group.iloc[:, 2].plot(kind='hist', bins=30, ax=axes[1], alpha=0.6, color='green')
        axes[1].set_title(f'Subfolder {subfolder} - Column 3')
        axes[1].set_xlabel('Value')
        axes[1].set_ylabel('Frequency')

        # Save the figure as an image
        image_path = os.path.join(temp_image_dir, f'subfolder_{subfolder}.png')
        plt.savefig(image_path)
        plt.close(fig)

        # Load the image into openpyxl
        img = Image(image_path)

        # Create a new sheet for the subfolder
        ws = wb.create_sheet(title=f'Subfolder {subfolder}')

        # Write the group DataFrame to the sheet
        for r_idx, row in enumerate(dataframe_to_rows(group, index=False, header=True), 1):
            for c_idx, value in enumerate(row, 1):
                ws.cell(row=r_idx, column=c_idx, value=value)

        # Add the image to the worksheet
        ws.add_image(img, 'E2')

    # Save the Excel file
    wb.save(output_file)

    # Remove the temporary images directory
    for file in os.listdir(temp_image_dir):
        os.remove(os.path.join(temp_image_dir, file))
    os.rmdir(temp_image_dir)

# Example usage
if __name__ == "__main__":
    # Define the main folder path where the input Excel file is located and the output Excel file will be saved
    main_folder_path = '/Users/allisonpickle/Desktop/True Neuros/0_Mean_Threshold'

    # Plot histograms and save the results to the output Excel file
    plot_histograms(main_folder_path)

In [ ]:
import os
import shutil

def organize_png_files_by_third_element(folder_path):
    """
    Organize PNG files in the specified folder based on the third element of the filename.
    Create a new subfolder for each new third element encountered.

    Parameters
    ----------
    folder_path : str
        The path to the folder containing the PNG files.
    """
    # List all files in the folder
    files = [f for f in os.listdir(folder_path) if f.endswith('.png')]

    for file in files:
        # Split the filename by underscores
        elements = file.split('_')

        if len(elements) >= 3:
            # Get the third element of the filename
            third_element = elements[2]

            # Create a new subfolder for the third element if it doesn't exist
            subfolder_path = os.path.join(folder_path, third_element)
            if not os.path.exists(subfolder_path):
                os.makedirs(subfolder_path)

            # Move the file to the corresponding subfolder
            src_path = os.path.join(folder_path, file)
            dst_path = os.path.join(subfolder_path, file)
            shutil.move(src_path, dst_path)

# Example usage
folder_path = '/Users/allisonpickle/Desktop/Live Imaging/Processed Bright'
organize_png_files_by_third_element(folder_path)

In [ ]:
import os
import imageio

def create_gif_from_subfolders(folder_path):
    """
    Iterate through a folder of subfolders and compile all images in each subfolder into a new GIF file.
    Save the GIF file under the subfolder's name.

    Parameters
    ----------
    folder_path : str
        The path to the folder containing the subfolders with images.
    """
    # Iterate through all subfolders in the folder
    for subfolder_name in os.listdir(folder_path):
        subfolder_path = os.path.join(folder_path, subfolder_name)
        
        if os.path.isdir(subfolder_path):
            # List all image files in the subfolder
            image_files = [f for f in os.listdir(subfolder_path) if f.endswith('.png') or f.endswith('.jpg') or f.endswith('.jpeg')]

            # Sort the image files to maintain order
            image_files.sort()

            # Read images and compile them into a list
            images = []
            for image_file in image_files:
                image_path = os.path.join(subfolder_path, image_file)
                images.append(imageio.imread(image_path))

            # Create a GIF from the images
            gif_path = os.path.join(subfolder_path, f"{subfolder_name}.gif")
            imageio.mimsave(gif_path, images, duration=0.5)

# Example usage
create_gif_from_subfolders('/Users/allisonpickle/Desktop/Live Imaging/Processed Bright')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

def plot_aspect_ratio_orientation(df, save_path):
    """
    Plot the Aspect Ratio as the R and orientation angle as the theta in a polar plot,
    grouped by the designation in the first column.

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame containing the data with aspect ratios, orientation angles, and designations.
    save_path : str
        The path where the plot will be saved.
    """
    # Extract the designations, aspect ratios, and orientation angles
    designations = df.iloc[:, 0]
    aspect_ratios = df.iloc[:, 9]  # Column J (10th column, index 9)
    orientation_angles = df.iloc[:, 10]  # Column K (11th column, index 10)

    # Convert orientation angles to radians
    theta = np.radians(orientation_angles)
    
    # Calculate R as aspect ratio - 1
    R = aspect_ratios - 1

    # Create a polar plot
    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)

    # Get unique designations for grouping
    unique_designations = designations.unique()
    colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_designations)))

    # Plot each group with a different color
    for i, designation in enumerate(unique_designations):
        mask = designations == designation
        ax.scatter(theta[mask], R[mask], label=designation, color=colors[i], s=15)  # Adjust marker size as needed

    ax.set_title('Polar Plot of Cell Orientation and Aspect Ratio')
    ax.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1))
    plt.savefig(os.path.join(save_path, 'Polar_Plot.png'))
    plt.show()

def plot_first_quadrant(df, save_path):
    """
    Plot the first quadrant (0 to 90 degrees) of the Aspect Ratio as the R and orientation angle as the theta in a polar plot,
    grouped by the designation in the first column.

    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame containing the data with aspect ratios, orientation angles, and designations.
    save_path : str
        The path where the plot will be saved.
    """
    # Extract the designations, aspect ratios, and orientation angles
    designations = df.iloc[:, 0]
    aspect_ratios = df.iloc[:, 9]  # Column J (10th column, index 9)
    orientation_angles = df.iloc[:, 10]  # Column K (11th column, index 10)

    # Convert orientation angles to radians
    theta = np.radians(orientation_angles)
    
    # Calculate R as aspect ratio - 1
    R = aspect_ratios - 1

    # Create a polar plot for the first quadrant
    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)

    # Get unique designations for grouping
    unique_designations = designations.unique()
    colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_designations)))

    # Plot each group with a different color
    for i, designation in enumerate(unique_designations):
        mask = designations == designation
        ax.scatter(theta[mask], R[mask], label=designation, color=colors[i], s=20)  # Adjust marker size as needed

    ax.set_ylim(0, max(R))
    ax.set_xlim(0, np.radians(90))
    ax.set_title('Polar Plot of Cell Orientation and Aspect Ratio (0 to 90°)')
    ax.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1))
    plt.savefig(os.path.join(save_path, 'Polar_Plot_First_Quadrant.png'))
    plt.show()

# Read data from Excel file
excel_file = '/Users/allisonpickle/Desktop/250131_Si Ridge/To Process/Split Channels/C_2 Microglia/0_Mean_Threshold/Calculations.xlsx'
df = pd.read_excel(excel_file)

# Get the folder path of the Excel file
folder_path = os.path.dirname(excel_file)

# Plot the full data
plot_aspect_ratio_orientation(df, folder_path)

# Plot the first quadrant
plot_first_quadrant(df, folder_path)

In [31]:
import numpy as np
from skan import Skeleton, summarize
from skimage import io, morphology
import pandas as pd
import os
import matplotlib.pyplot as plt

def analyze_skeleton_orientation(image_path, save_visualization=True):
    """
    Analyze a binary image (thresholded or skeletonized) and extract branch orientation angles using skan.
    Automatically skeletonizes the image if needed.
    
    Parameters
    ----------
    image_path : str
        Path to the binary image (will be skeletonized automatically).
    save_visualization : bool
        Whether to save a visualization of the skeleton with branches traced.
    
    Returns
    -------
    dict
        Dictionary containing:
        - 'mean_angle': Mean orientation angle across all branches (0-90°)
        - 'median_angle': Median orientation angle
        - 'all_angles': List of all branch angles
        - 'branch_lengths': List of branch lengths
        - 'num_branches': Total number of branches detected
    """
    print(f"\nDEBUG: Loading image from: {image_path}")
    
    # Load the binary image
    try:
        binary_image = io.imread(image_path, as_gray=True)
        print(f"DEBUG: Image loaded. Shape: {binary_image.shape}, dtype: {binary_image.dtype}")
        print(f"DEBUG: Image value range: {binary_image.min()} to {binary_image.max()}")
    except Exception as e:
        print(f"ERROR: Failed to load image: {e}")
        return None
    
    # Ensure binary
    binary_image = binary_image > 0.5
    print(f"DEBUG: After thresholding - True pixels: {np.sum(binary_image)}, False pixels: {np.sum(~binary_image)}")
    
    if np.sum(binary_image) == 0:
        print("ERROR: Image is completely black after thresholding!")
        return None
    
    # Skeletonize the image
    print("DEBUG: Skeletonizing...")
    skeleton_image = morphology.skeletonize(binary_image)
    print(f"DEBUG: Skeleton created - True pixels: {np.sum(skeleton_image)}")
    
    if np.sum(skeleton_image) == 0:
        print("ERROR: Skeleton is empty!")
        return None
    
    # Create skeleton object
    print("DEBUG: Creating Skeleton object...")
    skeleton = Skeleton(skeleton_image)
    
    # Get branch data
    print("DEBUG: Analyzing branches...")
    branch_data = summarize(skeleton)
    print(f"DEBUG: Branch data shape: {branch_data.shape}")
    print(f"DEBUG: Branch data columns: {branch_data.columns.tolist()}")
    
    if len(branch_data) == 0:
        print(f"WARNING: No branches detected in {image_path}")
        return None
    
    # Extract angles from branch data
    angles = []
    lengths = []
    all_coords = []
    
    for idx, row in branch_data.iterrows():
        # Get the coordinates of the branch
        coords = skeleton.path_coordinates(idx)
        
        if len(coords) < 2:
            print(f"DEBUG: Skipping branch {idx} - only {len(coords)} coordinates")
            continue
            
        # Calculate the angle using the first and last points of the branch
        start_point = coords[0]
        end_point = coords[-1]
        
        # Calculate angle
        dy = end_point[0] - start_point[0]  # row difference
        dx = end_point[1] - start_point[1]  # column difference
        
        angle_rad = np.arctan2(dy, dx)
        angle_deg = np.degrees(angle_rad)
        
        # Normalize to 0-180° range
        if angle_deg < 0:
            angle_deg += 180
        
        # Mirror to 0-90° range (orientation only, not direction)
        if angle_deg > 90:
            angle_deg = 180 - angle_deg
        
        angles.append(angle_deg)
        lengths.append(row['branch-distance'])
        all_coords.append(coords)
    
    print(f"DEBUG: Extracted {len(angles)} valid branches")
    
    if len(angles) == 0:
        print("WARNING: No valid branches found after processing")
        return None
    
    # Calculate statistics
    mean_angle = np.mean(angles)
    median_angle = np.median(angles)
    weighted_mean_angle = np.average(angles, weights=lengths)
    
    print(f"DEBUG: Mean angle: {mean_angle:.2f}°, Median: {median_angle:.2f}°, Weighted: {weighted_mean_angle:.2f}°")
    
    # Create visualization if requested
    if save_visualization:
        visualize_skeleton_analysis(
            binary_image, 
            skeleton_image, 
            all_coords, 
            angles, 
            median_angle,
            image_path
        )
    
    return {
        'mean_angle': mean_angle,
        'median_angle': median_angle,
        'all_angles': angles,
        'branch_lengths': lengths,
        'num_branches': len(angles),
        'weighted_mean_angle': weighted_mean_angle
    }

def visualize_skeleton_analysis(original_image, skeleton_image, branch_coords, angles, median_angle, image_path):
    """
    Create and save a visualization showing the original image, skeleton with traced branches,
    and the median angle overlay.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    
    # Left panel: Original image with skeleton overlay
    axes[0].imshow(original_image, cmap='gray')
    axes[0].imshow(skeleton_image, cmap='hot', alpha=0.5)
    axes[0].set_title('Original + Skeleton Overlay')
    axes[0].axis('off')
    
    # Right panel: Skeleton with traced branches and angle overlay
    axes[1].imshow(skeleton_image, cmap='gray')
    
    # Plot each branch with color-coded by angle
    cmap = plt.cm.rainbow
    norm = plt.Normalize(vmin=0, vmax=90)
    
    for coords, angle in zip(branch_coords, angles):
        color = cmap(norm(angle))
        axes[1].plot(coords[:, 1], coords[:, 0], color=color, linewidth=2, alpha=0.7)
    
    # Add colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=axes[1], fraction=0.046, pad=0.04)
    cbar.set_label('Branch Angle (°)', rotation=270, labelpad=15)
    
    # Calculate center of image for angle line
    center_y, center_x = np.array(skeleton_image.shape) / 2
    line_length = min(skeleton_image.shape) * 0.3
    
    # Draw median angle line
    # Convert angle to radians - note: we need to account for image coordinates
    # In image coordinates: 0° = right (positive x), 90° = down (positive y)
    # But we calculated angles as standard math coordinates where 0° = right, 90° = up
    median_rad = np.radians(median_angle)
    end_x = center_x + line_length * np.cos(median_rad)
    end_y = center_y + line_length * np.sin(median_rad)  # Changed from minus to plus
    
    # Draw line in both directions from center
    start_x = center_x - line_length * np.cos(median_rad)
    start_y = center_y - line_length * np.sin(median_rad)
    
    axes[1].plot([start_x, end_x], [start_y, end_y], 'r-', linewidth=3, label=f'Median: {median_angle:.1f}°')
    axes[1].plot(center_x, center_y, 'ro', markersize=8)
    
    axes[1].set_title(f'Branch Traces (Median Angle: {median_angle:.1f}°)')
    axes[1].legend(loc='upper right')
    axes[1].axis('off')
    
    # Save the figure in a "Skeleton_Visualizations" subfolder
    folder_path_parent = os.path.dirname(image_path)
    viz_folder = os.path.join(folder_path_parent, "Skeleton_Visualizations")
    os.makedirs(viz_folder, exist_ok=True)
    
    image_name = os.path.splitext(os.path.basename(image_path))[0]
    save_path = os.path.join(viz_folder, f"{image_name}_analysis.png")
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"Saved visualization to {save_path}")

def process_skeleton_images_in_folder(folder_path):
    """
    Process all binary images directly in the specified folder (no subfolders).
    """
    print(f"\n{'='*60}")
    print(f"Starting to process folder: {folder_path}")
    print(f"{'='*60}\n")
    
    all_results = []
    
    # Get all image files directly in the folder (no subdirectories)
    image_files = [f for f in os.listdir(folder_path) 
                   if os.path.isfile(os.path.join(folder_path, f)) 
                   and f.endswith(('.tif', '.png', '.jpg', '.jpeg', '.tiff'))]
    
    print(f"Found {len(image_files)} images in folder\n")
    
    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)
        image_name = os.path.splitext(image_file)[0]
        
        print(f"\n>>> Processing: {image_name}")
        
        # Analyze skeleton with visualization
        result = analyze_skeleton_orientation(image_path, save_visualization=True)
        
        if result is not None:
            all_results.append({
                'Image Name': image_name,
                'Mean Angle': result['mean_angle'],
                'Median Angle': result['median_angle'],
                'Weighted Mean Angle': result['weighted_mean_angle'],
                'Number of Branches': result['num_branches'],
                'Total Branch Length': sum(result['branch_lengths'])
            })
            print(f"✓ Successfully processed {image_name}")
        else:
            print(f"✗ Failed to process {image_name}")
    
    if len(all_results) == 0:
        print("\n" + "="*60)
        print("ERROR: No images were successfully processed!")
        print("="*60)
        return None
    
    # Save results to Excel
    df = pd.DataFrame(all_results)
    output_file = os.path.join(folder_path, "Skeleton_Orientation_Analysis.xlsx")
    df.to_excel(output_file, index=False)
    print(f"\n{'='*60}")
    print(f"✓ Results saved to: {output_file}")
    print(f"{'='*60}\n")
    
    return df

# Example usage
if __name__ == "__main__":
    folder_path = '/Users/allisonpickle/Desktop/Test/0_Threshold'
    results_df = process_skeleton_images_in_folder(folder_path)
    if results_df is not None:
        print(results_df)


Starting to process folder: /Users/allisonpickle/Desktop/Test/0_Threshold

Found 1 images in folder


>>> Processing: unnamed-1

DEBUG: Loading image from: /Users/allisonpickle/Desktop/Test/0_Threshold/unnamed-1.tif
DEBUG: Image loaded. Shape: (247, 362), dtype: uint8
DEBUG: Image value range: 0 to 255
DEBUG: After thresholding - True pixels: 18055, False pixels: 71359
DEBUG: Skeletonizing...
DEBUG: Skeleton created - True pixels: 5245
DEBUG: Creating Skeleton object...
DEBUG: Analyzing branches...
DEBUG: Branch data shape: (480, 16)
DEBUG: Branch data columns: ['skeleton-id', 'node-id-src', 'node-id-dst', 'branch-distance', 'branch-type', 'mean-pixel-value', 'stdev-pixel-value', 'image-coord-src-0', 'image-coord-src-1', 'image-coord-dst-0', 'image-coord-dst-1', 'coord-src-0', 'coord-src-1', 'coord-dst-0', 'coord-dst-1', 'euclidean-distance']
DEBUG: Extracted 480 valid branches
DEBUG: Mean angle: 45.92°, Median: 45.00°, Weighted: 44.43°


/var/folders/9q/b735k4x1335_j77np9rz8xlc0000gn/T/ipykernel_305/1264029011.py:64: VisibleDeprecationWarning: separator in column name will change to _ in version 0.13; to silence this warning, use `separator='-'` to maintain current behavior and use `separator='_'` to switch to the new default behavior.
  branch_data = summarize(skeleton)


Saved visualization to /Users/allisonpickle/Desktop/Test/0_Threshold/Skeleton_Visualizations/unnamed-1_analysis.png
✓ Successfully processed unnamed-1

✓ Results saved to: /Users/allisonpickle/Desktop/Test/0_Threshold/Skeleton_Orientation_Analysis.xlsx

  Image Name  Mean Angle  Median Angle  Weighted Mean Angle  \
0  unnamed-1   45.916695          45.0            44.434775   

   Number of Branches  Total Branch Length  
0                 480          6056.880877  


add a Sholl like count for branching